In [1]:
import pandas as pd
from huggingface_hub import hf_hub_download

def load_table(filename):
    path = hf_hub_download(repo_id="hao-li/AIDev", filename=filename, repo_type="dataset")
    return pd.read_parquet(path)

prs      = load_table("pull_request.parquet")
commits  = load_table("pr_commits.parquet")
reviews  = load_table("pr_reviews.parquet")
rev_cmts = load_table("pr_review_comments_v2.parquet")
timeline = load_table("pr_timeline.parquet")

In [ ]:
import sys
sys.path.insert(0, '.')
from helpers import build_pr_states, categorize

merged = prs[prs['merged_at'].notna()].copy()

pr_states = build_pr_states(reviews, merged['id'])

merged['review_category'] = merged['id'].apply(lambda pr_id: categorize(pr_id, pr_states))

revised_prs = merged[
    merged['review_category'].isin([
        'changes_requested_then_approved',
        'changes_requested_and_dismissed_then_approved'
    ])
].copy()

print(f"Problematic-Revised PRs (raw): {len(revised_prs):,}")
revised_prs['review_category'].value_counts()

In [ ]:
# Filter 1: Remove self-approvals — APPROVED review must be from a human User
human_approved_pr_ids = set(
    reviews[
        (reviews['state'] == 'APPROVED') &
        (reviews['user_type'] == 'User')
    ]['pr_id'].unique()
)

revised_prs = revised_prs[revised_prs['id'].isin(human_approved_pr_ids)]
print(f"After removing bot-approved PRs: {len(revised_prs):,}")

In [ ]:
# Filter 2: Remove PRs where CHANGES_REQUESTED came only from bots
human_cr_pr_ids = set(
    reviews[
        (reviews['state'] == 'CHANGES_REQUESTED') &
        (reviews['user_type'] == 'User')
    ]['pr_id'].unique()
)

revised_prs = revised_prs[revised_prs['id'].isin(human_cr_pr_ids)]
print(f"After requiring human CHANGES_REQUESTED: {len(revised_prs):,}")

In [ ]:
# For each surviving PR, reconstruct the revision cycle:
# reviews and commits ordered by timestamp

revised_prs['created_at'] = pd.to_datetime(revised_prs['created_at'])
revised_prs['merged_at']  = pd.to_datetime(revised_prs['merged_at'])

pr_reviews_sub = reviews[
    reviews['pr_id'].isin(revised_prs['id']) &
    (reviews['user_type'] == 'User')
][['pr_id', 'id', 'user', 'state', 'submitted_at']].copy()
pr_reviews_sub['submitted_at'] = pd.to_datetime(pr_reviews_sub['submitted_at'])

pr_commits_sub = commits[
    commits['pr_id'].isin(revised_prs['id'])
].merge(
    timeline[
        (timeline['pr_id'].isin(revised_prs['id'])) &
        (timeline['event'] == 'committed')
    ][['pr_id', 'commit_id', 'created_at']].rename(columns={'commit_id': 'sha'}),
    on=['pr_id', 'sha'], how='left'
).copy()
pr_commits_sub['created_at'] = pd.to_datetime(pr_commits_sub['created_at'])

print(f"Reviews (human, on revised PRs): {len(pr_reviews_sub):,}")
print(f"Commits (on revised PRs):        {len(pr_commits_sub):,}")

In [ ]:
# For each PR, count commits that came AFTER the first CHANGES_REQUESTED review
def count_revision_commits(pr_id):
    cr_reviews = pr_reviews_sub[
        (pr_reviews_sub['pr_id'] == pr_id) &
        (pr_reviews_sub['state'] == 'CHANGES_REQUESTED')
    ]
    if cr_reviews.empty:
        return 0
    first_cr_time = cr_reviews['submitted_at'].min()
    revision_commits = pr_commits_sub[
        (pr_commits_sub['pr_id'] == pr_id) &
        (pr_commits_sub['created_at'] > first_cr_time)
    ]
    return len(revision_commits)

revised_prs['revision_commit_count'] = revised_prs['id'].apply(count_revision_commits)

print(revised_prs['revision_commit_count'].describe())
print(f"\nPRs with 0 revision commits (timestamp gap): {(revised_prs['revision_commit_count'] == 0).sum()}")

In [ ]:
# Pull inline review comments from CHANGES_REQUESTED reviews
cr_review_ids = set(
    reviews[
        (reviews['pr_id'].isin(revised_prs['id'])) &
        (reviews['state'] == 'CHANGES_REQUESTED') &
        (reviews['user_type'] == 'User')
    ]['id'].unique()
)

cr_inline_comments = rev_cmts[
    rev_cmts['pull_request_review_id'].isin(cr_review_ids)
][['pull_request_review_id', 'user', 'path', 'diff_hunk', 'body', 'created_at']].copy()

# Join back to pr_id
review_id_to_pr = reviews.set_index('id')['pr_id'].to_dict()
cr_inline_comments['pr_id'] = cr_inline_comments['pull_request_review_id'].map(review_id_to_pr)

print(f"PRs with inline CR comments:  {cr_inline_comments['pr_id'].nunique():,}")
print(f"Total inline CR comments:     {len(cr_inline_comments):,}")

In [ ]:
# Summary of the clean Problematic-Revised subset
print("=== Problematic-Revised Clean Subset ===")
print(f"Total PRs:                          {len(revised_prs):,}")
print(f"  changes_requested_then_approved:  {(revised_prs['review_category'] == 'changes_requested_then_approved').sum():,}")
print(f"  CR + dismissed then approved:     {(revised_prs['review_category'] == 'changes_requested_and_dismissed_then_approved').sum():,}")
print(f"PRs with inline CR comments:        {cr_inline_comments['pr_id'].nunique():,}")
print(f"PRs with tracked revision commits:  {(revised_prs['revision_commit_count'] > 0).sum():,}")
print(f"Median revision commits per PR:     {revised_prs['revision_commit_count'].median()}")

In [ ]:
# Inspect a sample revision cycle end-to-end
sample_id = revised_prs[
    revised_prs['id'].isin(cr_inline_comments['pr_id'])
].sample(1, random_state=42)['id'].iloc[0]

print("=== PR ===")
print(prs[prs['id'] == sample_id][['id', 'title', 'agent', 'repo_url', 'created_at', 'merged_at']].to_string(index=False))

print("\n=== Human Reviews (ordered) ===")
print(
    pr_reviews_sub[pr_reviews_sub['pr_id'] == sample_id]
    .sort_values('submitted_at')[['state', 'user', 'submitted_at']]
    .to_string(index=False)
)

print("\n=== Commits ===")
print(
    pr_commits_sub[pr_commits_sub['pr_id'] == sample_id]
    .sort_values('created_at')[['sha', 'author', 'message', 'created_at']]
    .to_string(index=False)
)

print("\n=== Inline CR Comments ===")
print(
    cr_inline_comments[cr_inline_comments['pr_id'] == sample_id]
    [['path', 'body']]
    .to_string(index=False)
)